Notebook: 10_confidence_model.ipynb

Purpose: Train calibrated confidence prediction models using record-level evidence.

Inputs:
- confidence_features.parquet

Outputs:
- confidence_predictions.parquet

# 10 — Confidence Model Training

Train and calibrate classification models that estimate the probability a QT measurement is trustworthy using the fused evidence table.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
sys.path.insert(0, str(Path.cwd().parent / 'src'))

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
confidence = pd.read_parquet(artifacts_dir / 'confidence_features.parquet')
feature_columns = [
    'mean_hfn_index','std_hfn_index','max_hfn_index','p95_hfn_index',
    'mean_pli_index','std_pli_index','max_pli_index','p95_pli_index',
    'mean_clipping_ratio','std_clipping_ratio','max_clipping_ratio','p95_clipping_ratio',
    'mean_flatline_ratio','std_flatline_ratio','max_flatline_ratio','p95_flatline_ratio',
    'mean_signal_quality_score','std_signal_quality_score','max_signal_quality_score','p95_signal_quality_score',
    'mean_boundary_confidence','std_boundary_confidence','max_t_end_uncertainty_ms','p95_t_end_uncertainty_ms',
    'mean_t_end_ambiguity_score','max_t_end_ambiguity_score','mean_morphology_confidence',
    'mean_bsqi','min_bsqi','mean_wsqi','min_wsqi','mean_lead_agreement','worst_lead_agreement',
    'mean_beat_agreement','worst_beat_agreement','qt_variance_leads','qt_variance_beats',
]
cat_columns = ['diagnostic_class', 'dataset_origin']
for col in cat_columns:
    if col not in confidence.columns:
        confidence[col] = np.nan
confidence['confidence_label'] = (
    (confidence['mean_lead_agreement'] >= 0.7) &
    (confidence['mean_beat_agreement'] >= 0.7) &
    (confidence['mean_bsqi'] >= 0.6) &
    (confidence['mean_wsqi'] >= 0.6)
).astype(int)

X = confidence[feature_columns + cat_columns].copy()
y = confidence['confidence_label'].copy()

from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, feature_columns),
    ('cat', cat_transformer, cat_columns),
])

models = [('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42))]

try:
    import xgboost as xgb
    models.append(('XGBoost', xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)))
except Exception:
    pass

try:
    import lightgbm as lgb
    models.append(('LightGBM', lgb.LGBMClassifier(random_state=42)))
except Exception:
    pass

results = []
for name, model in models:
    if len(np.unique(y)) < 2:
        continue
    clf = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model),
    ])
    calibrated = CalibratedClassifierCV(clf, cv=3, method='isotonic')
    calibrated.fit(X, y)
    preds = calibrated.predict_proba(X)[:, 1]
    results.append((name, calibrated, preds))

if not results:
    raise RuntimeError('No confidence model could be trained.')

name, model, preds = results[0]
confidence['confidence_probability'] = preds
confidence['model_name'] = name
confidence['calibration_method'] = 'isotonic'
confidence['pipeline_version'] = 'v1.0.0'
confidence[['record_id','confidence_probability','model_name','calibration_method','pipeline_version']].to_parquet(artifacts_dir / 'confidence_predictions.parquet', index=False)
print('Wrote confidence_predictions.parquet')
